### Imports

In [1]:
import os
from pathlib import Path
import json
from tqdm import tqdm
import sys
import glob
import shutil
from pathlib import Path
from typing import List, Dict, Any, Set, Tuple
from tqdm import tqdm

sys.path.insert(0, "../../")
from config import DATASETS_PATH, SEED, IMG_SHAPE


### Functions

In [2]:
def preprocess_sources(
    images_paths: List[str], 
    annotations_paths: List[str]
) -> Tuple[Dict[str, Path], Dict[str, Path], Dict[Path, Dict]]:
    """Scans all source files once to create fast lookup dictionaries."""
    print("Pre-processing source files for fast lookups...")
    
    # 1. Map image filename to its full path
    image_name_to_path = {Path(p).name: Path(p) for p in images_paths}
    
    # 2. Map image filename to the JSON file that describes it
    image_name_to_json_path = {}
    
    # 3. Cache the content of each JSON file
    json_content_cache = {}

    for json_path_str in tqdm(annotations_paths, desc="Scanning annotations"):
        json_path = Path(json_path_str)
        if json_path not in json_content_cache:
            with open(json_path) as f:
                data = json.load(f)
                json_content_cache[json_path] = data
            for image in data.get('images', []):
                image_name_to_json_path[image['file_name']] = json_path
                
    return image_name_to_path, image_name_to_json_path, json_content_cache

# --- 2. Main Orchestration Function ---

def create_consolidated_dataset_revised(
    classes: List[str],
    tagged_crops: List[str],
    images_paths: List[str],
    annotations_paths: List[str],
    output_path_str: str
):
    """
    Creates a new, unified COCO dataset using an annotation-driven approach.
    """
    output_path = Path(output_path_str)
    images_dir = output_path / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)
    
    # Step 1: Pre-process all sources to build our lookup maps
    image_name_map, image_json_map, json_cache = preprocess_sources(images_paths, annotations_paths)
    
    # Step 2: Initialize structures for the new dataset
    final_annotations = []
    final_images = []
    final_categories = [{'id': i + 1, 'name': name, 'supercategory': 'none'} for i, name in enumerate(classes)]
    
    # --- Mappings to handle new, sequential IDs ---
    category_name_to_id = {cat['name']: cat['id'] for cat in final_categories}
    # Map original image filename to its new ID in our dataset
    image_name_to_new_id = {}
    
    next_ann_id = 1
    next_image_id = 1
    
    print(f"\nProcessing {len(tagged_crops)} tagged crops to build dataset...")
    for crop_path_str in tqdm(tagged_crops, desc="Building dataset"):
        crop_path = Path(crop_path_str)
        
        # --- A. Parse information from the crop file path ---
        try:
            class_name = crop_path.parent.name
            stem = crop_path.stem
            ann_id = int(stem.split('_')[-1])
            base_image_stem = '_'.join(stem.split('_')[:-1])
        except (ValueError, IndexError):
            print(f"⚠️ Warning: Could not parse file: {crop_path}. Skipping.")
            continue

        # Find the full name of the original image (e.g., 'imageA.jpg')
        original_image_name = next((name for name in image_name_map if Path(name).stem == base_image_stem), None)
        if not original_image_name:
            print(f"⚠️ Warning: No matching full image found for crop '{crop_path.name}'. Skipping.")
            continue
            
        # --- B. Use lookups to find the source data ---
        source_json_path = image_json_map.get(original_image_name)
        if not source_json_path:
            print(f"⚠️ Warning: No annotation file found for image '{original_image_name}'. Skipping.")
            continue
            
        source_data = json_cache[source_json_path]
        
        # Find the original annotation data using its ID
        source_ann = next((ann for ann in source_data['annotations'] if ann['id'] == ann_id), None)
        if not source_ann:
            print(f"⚠️ Warning: Annotation ID {ann_id} not found in '{source_json_path}'. Skipping.")
            continue
            
        # --- C. Add image to our new dataset (if it's the first time seeing it) ---
        new_image_id = image_name_to_new_id.get(original_image_name)
        if new_image_id is None:
            source_image_path = image_name_map[original_image_name]
            
            # 1. Assign a new sequential ID
            new_image_id = next_image_id
            image_name_to_new_id[original_image_name] = new_image_id
            next_image_id += 1
            
            # 2. Add the image entry to our final list
            source_image_obj = next(img for img in source_data['images'] if img['file_name'] == original_image_name)
            final_images.append({
                "id": new_image_id,
                "file_name": original_image_name,
                "width": source_image_obj.get('width'),
                "height": source_image_obj.get('height')
            })
            
            # 3. Copy the actual image file
            shutil.copy(source_image_path, images_dir)

        # --- D. Create and add the new annotation entry ---
        new_category_id = category_name_to_id.get(class_name)
        if new_category_id is None:
            print(f"⚠️ Warning: Crop class '{class_name}' not in provided class list. Skipping.")
            continue

        final_annotations.append({
            "id": next_ann_id,
            "image_id": new_image_id,
            "category_id": new_category_id,
            "bbox": source_ann['bbox'],
            "iscrowd": source_ann.get('iscrowd', 0),
            "area": source_ann.get('area', 0),
            "segmentation": source_ann.get('segmentation', [])
        })
        next_ann_id += 1
        
    # Step 3: Assemble and save the final dataset file
    unified_coco = {
        "info": {"description": "Consolidated Dataset"},
        "licenses": [],
        "images": final_images,
        "annotations": final_annotations,
        "categories": final_categories
    }

    final_json_path = output_path / 'annotations.json'
    print(f"\nSaving unified annotations to: {final_json_path}")
    with open(final_json_path, 'w') as f:
        json.dump(unified_coco, f, indent=4)
        
    print("\n✅ Dataset creation complete!")
    print(f"   - Total images: {len(final_images)}")
    print(f"   - Total annotations: {len(final_annotations)}")

#### Get files

In [7]:
# Get all unlabeled image paths
INA_ANNOTATION_PATH = os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'ina', 'annotated_coco.json')
INA_TAGGED_IMAGES_PATH = os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'ina', 'images')
INA_TAGGED_CROPS_PATH = os.path.join(DATASETS_PATH, 'cropped', 'annotated', 'ina')

ROBOFLOW_ANNOTATION_PATH = os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'roboflow_datasets', 'annotated_coco.json')
ROBOFLOW_TAGGED_IMAGES_PATH = os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'roboflow_datasets', 'images')
ROBOFLOW_TAGGED_CROPS_PATH = os.path.join(DATASETS_PATH, 'cropped', 'annotated', 'roboflow_datasets')

YOLO_DATASET_PATH = os.path.join(DATASETS_PATH, 'cropped', 'yolo2')

# classes = [os.path.basename(file) for file in glob.glob(INA_TAGGED_CROPS_PATH)]
classes = ['prophase', 'metaphase', 'anaphase', 'telophase', 'interphase']
tagged_ina_crops = glob.glob(os.path.join(INA_TAGGED_CROPS_PATH, '*/*'))
tagged_roboflow_crops = glob.glob(os.path.join(ROBOFLOW_TAGGED_CROPS_PATH, '*/*'))
tagged_crops = tagged_ina_crops + tagged_roboflow_crops

ina_images = glob.glob(os.path.join(INA_TAGGED_IMAGES_PATH, '*'))
roboflow_images = glob.glob(os.path.join(ROBOFLOW_TAGGED_IMAGES_PATH, '*'))
images = ina_images + roboflow_images

annotations_paths = [INA_ANNOTATION_PATH + ROBOFLOW_ANNOTATION_PATH]

### Run

In [8]:
create_consolidated_dataset_revised(
    classes=classes,
    tagged_crops=tagged_crops,
    images_paths=images,
    annotations_paths=annotations_paths,
    output_path_str=YOLO_DATASET_PATH
)

Pre-processing source files for fast lookups...


Scanning annotations:   0%|          | 0/1 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/processed/ina/annotated_coco.json/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/full_fov/processed/roboflow_datasets/annotated_coco.json'